# Chạy bù toàn bộ số liệu E1 còn thiếu: Baseline Val + EXP-1 (YOLOv8s) + EXP-3 (RT-DETR-L)
**Giao thức chung:** Dataset gốc (368 train / 65 val, 7 classes), `imgsz=640`, `seed=0`, `epochs=50`.

In [ ]:
# 1. Kiểm tra GPU T4 & cài ultralytics
!nvidia-smi
!pip install -q ultralytics

In [ ]:
# 2. Mount Drive, giải nén dataset gốc và tạo /content/data_helmet.yaml
from google.colab import drive
import os, shutil

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/Traffic_Project/all_datasets.zip"
EXTRACT_DIR = "/content/dataset"
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"Không thấy {ZIP_PATH}. Hãy đảm bảo lối tắt Traffic_Project đã nằm trong MyDrive.")

!mkdir -p "{EXTRACT_DIR}"
!unzip -q -o "{ZIP_PATH}" -d "{EXTRACT_DIR}"

base = "/content/dataset/content/smart-helmet-detection-using-dl/data"
if os.path.exists(f"{base}/vaid") and not os.path.exists(f"{base}/valid"):
    shutil.move(f"{base}/vaid", f"{base}/valid")

yaml_text = f"""train: {base}/train
val: {base}/valid
nc: 7
names:
- driver_with_helmet
- bike
- driver
- passenger_with_helemt
- passenger
- driver_without_helmet
- passenger_without_helemt
"""
with open("/content/data_helmet.yaml", "w") as f:
    f.write(yaml_text)

print("Sẵn sàng data_helmet.yaml | train:", os.path.exists(f"{base}/train"), "| val:", os.path.exists(f"{base}/valid"))

In [ ]:
# 3. Đánh giá thống nhất Baseline YOLOv8n (helmet_best.pt) để lấy đủ Class 3/5/6
import os
from ultralytics import YOLO
from google.colab import files

DRIVE_BASELINE = "/content/drive/MyDrive/Traffic_Project/helmet_detection_v1/weights/best.pt"
if os.path.exists(DRIVE_BASELINE):
    baseline_pt = DRIVE_BASELINE
    print("Dùng baseline từ Drive:", baseline_pt)
else:
    print("Không thấy best.pt trên Drive -> Vui lòng chọn file models/weights_helmet/helmet_best.pt từ máy bạn:")
    uploaded = files.upload()
    baseline_pt = list(uploaded.keys())[0]

def summarize_metrics(label, m, weights_path, train_min=None):
    box = m.box
    c3 = m.class_result(3)
    c5 = m.class_result(5)
    c6 = m.class_result(6)
    size_mb = round(os.path.getsize(weights_path) / (1024 * 1024), 2)
    info = {
        "model": label,
        "P": round(float(box.mp), 3),
        "R": round(float(box.mr), 3),
        "mAP50": round(float(box.map50), 3),
        "mAP50_95": round(float(box.map), 3),
        "cls3_PR": f"{c3[0]:.3f} / {c3[1]:.3f}",
        "cls5_PR": f"{c5[0]:.3f} / {c5[1]:.3f}",
        "cls6_PR": f"{c6[0]:.3f} / {c6[1]:.3f}",
        "size_mb": size_mb,
        "train_min": train_min,
    }
    print(f"=== SUMMARY {label} ===", info)
    return info

m_base = YOLO(baseline_pt).val(data="/content/data_helmet.yaml", imgsz=640, split="val", project="/content/runs/e1", name="baseline_val", exist_ok=True)
summary_all = []
summary_all.append(summarize_metrics("Baseline YOLOv8n (95ep)", m_base, baseline_pt, train_min=10.17))

In [ ]:
# 4. EXP-1: YOLOv8s (50 epochs, imgsz=640, batch=16, seed=0)
import time
from ultralytics import YOLO

t0 = time.time()
YOLO("yolov8s.pt").train(
    data="/content/data_helmet.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    seed=0,
    project="/content/runs/e1",
    name="exp1_yolov8s",
    exist_ok=True,
)
exp1_min = round((time.time() - t0) / 60, 2)
print("EXP1_TRAIN_MINUTES:", exp1_min)

exp1_best = "/content/runs/e1/exp1_yolov8s/weights/best.pt"
m_exp1 = YOLO(exp1_best).val(data="/content/data_helmet.yaml", imgsz=640, split="val", project="/content/runs/e1", name="exp1_val", exist_ok=True)
summary_all.append(summarize_metrics("EXP-1 YOLOv8s (50ep)", m_exp1, exp1_best, train_min=exp1_min))

In [ ]:
# 5. EXP-3: RT-DETR-L (50 epochs, imgsz=640, seed=0, batch=16 -> fallback batch=8 nếu thiếu VRAM)
import time, torch
from ultralytics import RTDETR

t0 = time.time()
batch_used = 16
try:
    RTDETR("rtdetr-l.pt").train(
        data="/content/data_helmet.yaml",
        epochs=50,
        imgsz=640,
        batch=16,
        seed=0,
        project="/content/runs/e1",
        name="exp3_rtdetr",
        exist_ok=True,
    )
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    batch_used = 8
    print("T4 hết VRAM với batch=16 -> tự động chuyển sang batch=8")
    RTDETR("rtdetr-l.pt").train(
        data="/content/data_helmet.yaml",
        epochs=50,
        imgsz=640,
        batch=8,
        seed=0,
        project="/content/runs/e1",
        name="exp3_rtdetr",
        exist_ok=True,
    )

exp3_min = round((time.time() - t0) / 60, 2)
print(f"EXP3_TRAIN_MINUTES (batch={batch_used}):", exp3_min)

exp3_best = "/content/runs/e1/exp3_rtdetr/weights/best.pt"
m_exp3 = RTDETR(exp3_best).val(data="/content/data_helmet.yaml", imgsz=640, split="val", project="/content/runs/e1", name="exp3_val", exist_ok=True)
summary_all.append(summarize_metrics(f"EXP-3 RT-DETR-L (50ep, batch={batch_used})", m_exp3, exp3_best, train_min=exp3_min))

In [ ]:
# 6. In bảng tổng hợp + lưu file summary + nén toàn bộ artifact tải về máy
import json
from google.colab import files

with open("/content/runs/e1/e1_summary_exp1_exp3.json", "w", encoding="utf-8") as f:
    json.dump(summary_all, f, indent=2, ensure_ascii=False)

print("\n=== BẢNG TỔNG HỢP (COPY ĐOẠN NÀY GỬI CHO ASSISTANT) ===")
for row in summary_all:
    print(json.dumps(row, ensure_ascii=False))

!zip -rq /content/exp1_exp3_results.zip /content/runs/e1
!cp /content/exp1_exp3_results.zip /content/drive/MyDrive/exp1_exp3_results.zip
files.download("/content/exp1_exp3_results.zip")